# Phase 5: Geographic and Store Analysis

This notebook analyzes country performance, store performance, store rankings, and store growth trends.

## 1. Import libraries and load the cleaned dataset

In [1]:
from pathlib import Path
import pandas as pd

current_folder = Path.cwd()

if current_folder.name == 'notebooks':
    project_folder = current_folder.parent
else:
    project_folder = current_folder

cleaned_file = project_folder / 'data' / 'processed' / 'cleaned_sales.csv'
sales = pd.read_csv(cleaned_file)

print('Rows loaded:', len(sales))

Rows loaded: 62884


## 2. Analyze country performance

Country performance is based on the customer's country.

In [2]:
country_performance = (
    sales.groupby('Country_Customer', as_index=False)
    .agg(
        Revenue=('Sales USD', 'sum'),
        Profit=('Profit USD', 'sum'),
        Customers=('CustomerKey', 'nunique'),
        Orders=('Order Number', 'nunique'),
    )
    .rename(columns={'Country_Customer': 'Country'})
)

country_performance['Profit Margin %'] = (
    country_performance['Profit']
    / country_performance['Revenue']
    * 100
)

country_performance[['Revenue', 'Profit', 'Profit Margin %']] = (
    country_performance[['Revenue', 'Profit', 'Profit Margin %']].round(2)
)

country_performance = country_performance.sort_values('Revenue', ascending=False)
display(country_performance)

,Country,Revenue,Profit,Customers,Orders,Profit Margin %
7,United States,29871631.17,17490924.05,5706,14221,58.55
6,United Kingdom,7084088.12,4133800.81,1570,3421,58.35
3,Germany,5414149.80,3185916.46,1150,2440,58.84
1,Canada,4724334.63,2755738.22,1179,2281,58.33
0,Australia,2708137.61,1597300.18,780,1182,58.98
4,Italy,2475645.77,1450851.09,530,1101,58.60
5,Netherlands,1962154.27,1154644.08,534,976,58.85
2,France,1515338.22,893513.49,438,704,58.96


## 3. Answer the country questions

In [3]:
best_country = country_performance.loc[country_performance['Revenue'].idxmax()]
worst_country = country_performance.loc[country_performance['Revenue'].idxmin()]
highest_margin_country = country_performance.loc[country_performance['Profit Margin %'].idxmax()]

print('Best-performing country:', best_country['Country'])
print(f'Revenue: ${best_country["Revenue"]:,.2f}')
print()
print('Worst-performing country:', worst_country['Country'])
print(f'Revenue: ${worst_country["Revenue"]:,.2f}')
print()
print('Country with highest profit margin:', highest_margin_country['Country'])
print(f'Profit margin: {highest_margin_country["Profit Margin %"]:.2f}%')

Best-performing country: United States
Revenue: $29,871,631.17

Worst-performing country: France
Revenue: $1,515,338.22

Country with highest profit margin: Australia
Profit margin: 58.98%


## 4. Analyze store performance

In [4]:
store_performance = (
    sales.groupby(['StoreKey', 'Country_Store', 'State_Store'], as_index=False)
    .agg(
        Revenue=('Sales USD', 'sum'),
        Profit=('Profit USD', 'sum'),
        Orders=('Order Number', 'nunique'),
    )
)

store_performance['Profit Margin %'] = (
    store_performance['Profit']
    / store_performance['Revenue']
    * 100
)

store_performance[['Revenue', 'Profit', 'Profit Margin %']] = (
    store_performance[['Revenue', 'Profit', 'Profit Margin %']].round(2)
)

store_performance = store_performance.sort_values('Revenue', ascending=False)
display(store_performance)

,StoreKey,Country_Store,State_Store,Revenue,Profit,Orders,Profit Margin %
0,0,Online,Online,11404324.63,6672692.64,5580,58.51
48,55,United States,Nevada,1417885.41,830551.75,622,58.58
44,50,United States,Kansas,1394738.06,819659.12,605,58.77
47,54,United States,Nebraska,1384396.24,810300.06,629,58.53
7,9,Canada,Northwest Territories,1336150.06,774757.55,658,57.98
50,57,United States,New Mexico,1325611.89,784095.76,605,59.15
40,45,United States,Connecticut,1318787.92,761271.82,617,57.73
52,61,United States,South Carolina,1305684.43,764997.20,624,58.59
51,59,United States,Oregon,1302272.44,754702.32,619,57.95
55,64,United States,Washington DC,1259350.98,745283.99,566,59.18


## 5. Find the top 10 stores

In [5]:
top_10_stores = store_performance.head(10)
display(top_10_stores)

,StoreKey,Country_Store,State_Store,Revenue,Profit,Orders,Profit Margin %
0,0,Online,Online,11404324.63,6672692.64,5580,58.51
48,55,United States,Nevada,1417885.41,830551.75,622,58.58
44,50,United States,Kansas,1394738.06,819659.12,605,58.77
47,54,United States,Nebraska,1384396.24,810300.06,629,58.53
7,9,Canada,Northwest Territories,1336150.06,774757.55,658,57.98
50,57,United States,New Mexico,1325611.89,784095.76,605,59.15
40,45,United States,Connecticut,1318787.92,761271.82,617,57.73
52,61,United States,South Carolina,1305684.43,764997.20,624,58.59
51,59,United States,Oregon,1302272.44,754702.32,619,57.95
55,64,United States,Washington DC,1259350.98,745283.99,566,59.18


## 6. Find the 10 lowest-performing stores

In [6]:
lowest_10_stores = store_performance.tail(10).sort_values('Revenue')
display(lowest_10_stores)

,StoreKey,Country_Store,State_Store,Revenue,Profit,Orders,Profit Margin %
2,2,Australia,Northern Territory,15175.99,9512.61,4,62.68
11,14,France,Franche-Comté,105714.05,63591.55,57,60.15
10,13,France,Corse,150925.12,88957.65,80,58.94
14,17,France,Martinique,159607.50,92449.65,79,57.92
9,12,France,Basse-Normandie,183091.04,107084.53,90,58.49
24,28,Italy,Caltanissetta,187109.49,108108.47,82,57.78
13,16,France,Limousin,199009.69,119915.80,91,60.26
12,15,France,La Réunion,205119.67,118097.69,93,57.58
15,18,France,Mayotte,226078.88,135077.79,81,59.75
1,1,Australia,Australian Capital Territory,243029.93,141355.64,111,58.16


## 7. Analyze yearly store growth

In [7]:
store_growth = (
    sales.groupby(['StoreKey', 'Country_Store', 'State_Store', 'Year'], as_index=False)
    .agg(Revenue=('Sales USD', 'sum'))
    .sort_values(['StoreKey', 'Year'])
)

store_growth['Previous Year Revenue'] = (
    store_growth.groupby('StoreKey')['Revenue'].shift(1)
)

store_growth['Growth %'] = (
    (store_growth['Revenue'] - store_growth['Previous Year Revenue'])
    / store_growth['Previous Year Revenue']
    * 100
)

store_growth[['Revenue', 'Previous Year Revenue', 'Growth %']] = (
    store_growth[['Revenue', 'Previous Year Revenue', 'Growth %']].round(2)
)

display(store_growth)

,StoreKey,Country_Store,State_Store,Year,Revenue,Previous Year Revenue,Growth %
0,0,Online,Online,2016,1169315.86,NaN,NaN
1,0,Online,Online,2017,1388497.44,1169315.86,18.74
2,0,Online,Online,2018,2582287.13,1388497.44,85.98
3,0,Online,Online,2019,3908495.70,2582287.13,51.36
4,0,Online,Online,2020,2069950.90,3908495.70,-47.04
...,...,...,...,...,...,...,...
316,66,United States,Wyoming,2017,139758.34,140542.46,-0.56
317,66,United States,Wyoming,2018,274647.79,139758.34,96.52
318,66,United States,Wyoming,2019,445048.69,274647.79,62.04
319,66,United States,Wyoming,2020,205111.88,445048.69,-53.91


## 8. Save the country and store datasets

In [8]:
processed_folder = project_folder / 'data' / 'processed'
country_file = processed_folder / 'country_performance.csv'
store_file = processed_folder / 'store_performance.csv'
store_growth_file = processed_folder / 'store_growth.csv'

country_performance.to_csv(country_file, index=False)
store_performance.to_csv(store_file, index=False)
store_growth.to_csv(store_growth_file, index=False)

print('Saved:', country_file)
print('Saved:', store_file)
print('Saved:', store_growth_file)

Saved: d:\Desktop\Pune Internship\Global-Electronics-Retail-Sales-Analysis-and-Business-Intelligence-Dashboard\data\processed\country_performance.csv
Saved: d:\Desktop\Pune Internship\Global-Electronics-Retail-Sales-Analysis-and-Business-Intelligence-Dashboard\data\processed\store_performance.csv
Saved: d:\Desktop\Pune Internship\Global-Electronics-Retail-Sales-Analysis-and-Business-Intelligence-Dashboard\data\processed\store_growth.csv


## 9. Display the Phase 5 answers

In [9]:
print('PHASE 5 ANSWERS')
print('-' * 50)
print(f'Best-performing country: {best_country["Country"]}')
print(f'Worst-performing country: {worst_country["Country"]}')
print(f'Country with highest profit margin: {highest_margin_country["Country"]}')
print(f'Top store: {int(top_10_stores.iloc[0]["StoreKey"])}')
print(f'Lowest-performing store: {int(lowest_10_stores.iloc[0]["StoreKey"])}')
display(top_10_stores)
display(lowest_10_stores)

PHASE 5 ANSWERS
--------------------------------------------------
Best-performing country: United States
Worst-performing country: France
Country with highest profit margin: Australia
Top store: 0
Lowest-performing store: 2


,StoreKey,Country_Store,State_Store,Revenue,Profit,Orders,Profit Margin %
0,0,Online,Online,11404324.63,6672692.64,5580,58.51
48,55,United States,Nevada,1417885.41,830551.75,622,58.58
44,50,United States,Kansas,1394738.06,819659.12,605,58.77
47,54,United States,Nebraska,1384396.24,810300.06,629,58.53
7,9,Canada,Northwest Territories,1336150.06,774757.55,658,57.98
50,57,United States,New Mexico,1325611.89,784095.76,605,59.15
40,45,United States,Connecticut,1318787.92,761271.82,617,57.73
52,61,United States,South Carolina,1305684.43,764997.20,624,58.59
51,59,United States,Oregon,1302272.44,754702.32,619,57.95
55,64,United States,Washington DC,1259350.98,745283.99,566,59.18


,StoreKey,Country_Store,State_Store,Revenue,Profit,Orders,Profit Margin %
2,2,Australia,Northern Territory,15175.99,9512.61,4,62.68
11,14,France,Franche-Comté,105714.05,63591.55,57,60.15
10,13,France,Corse,150925.12,88957.65,80,58.94
14,17,France,Martinique,159607.50,92449.65,79,57.92
9,12,France,Basse-Normandie,183091.04,107084.53,90,58.49
24,28,Italy,Caltanissetta,187109.49,108108.47,82,57.78
13,16,France,Limousin,199009.69,119915.80,91,60.26
12,15,France,La Réunion,205119.67,118097.69,93,57.58
15,18,France,Mayotte,226078.88,135077.79,81,59.75
1,1,Australia,Australian Capital Territory,243029.93,141355.64,111,58.16
